# Milestone 1 — Cora + label and 1-WL partitions

Load Cora, build the *label partition* (7 cells, $e_C = 0$ everywhere) and the *1-WL partition* at depths $L \in \{1,2,3\}$. Histogram per-cell $e_C$.

## Setup

Resolve `onboarding/projects/` on `sys.path`, attach the reflection log, and import the canonical references (used only for spot-checks; the student version derives its own implementations).

In [1]:
import os, sys
from pathlib import Path
_here = Path(os.getcwd()).resolve()
for _p in [_here, *_here.parents]:
    if (_p / 'shared' / '__init__.py').exists() and _p.name == 'projects':
        _PROJECTS = _p
        break
else:
    raise RuntimeError('could not locate onboarding/projects/')
_REPO = _PROJECTS.parent.parent
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

import numpy as np
import matplotlib
matplotlib.use('Agg')   # headless for CI
import matplotlib.pyplot as plt

from onboarding.projects import reflect
reflect.start(notebook='m1', store=_PROJECTS / '.reflection.jsonl')
print(f'[setup] projects root: {_PROJECTS}')


[setup] projects root: /Users/elouafiq/workdir/gnn_express/onboarding/projects


In [2]:
from onboarding.projects.shared import (
    Partition, label_partition, wl_partition,
    hbin, hbin_inverse, upper, lower, slack, verify, bracket_of,
    sum_partition, mean_partition, max_partition,
)
print('shared modules imported')


shared modules imported


In [3]:
from torch_geometric.datasets import Planetoid
_CACHE = Path.home() / '.cache' / 'planetoid'
_CACHE.mkdir(parents=True, exist_ok=True)
_cora = Planetoid(root=str(_CACHE / 'Cora'), name='Cora')[0]
n = int(_cora.num_nodes)
m = int(_cora.num_edges)
K = int(_cora.y.max().item()) + 1
print(f'Cora: n={n}, m={m}, classes={K}, x.shape={tuple(_cora.x.shape)}')


Cora: n=2708, m=10556, classes=7, x.shape=(2708, 1433)


## Label partition
7 cells, one per class, $e_C = 0$ by construction.

In [4]:
P_lbl = label_partition(_cora)
print(f'label partition: {len(P_lbl.cells)} cells; e_C={P_lbl.e}')


label partition: 7 cells; e_C=[0. 0. 0. 0. 0. 0. 0.]


In [5]:
assert len(P_lbl.cells) == K, f'M1: label partition has {len(P_lbl.cells)} cells, expected {K}'
assert all(e == 0 for e in P_lbl.e), f'M1: label partition has nonzero e_C: {P_lbl.e}'
assert abs(sum(P_lbl.q) - 1.0) < 1e-12
print('[GATE OK] M1.label: 7 cells, all e_C=0, q sums to 1')


[GATE OK] M1.label: 7 cells, all e_C=0, q sums to 1


## 1-WL partitions at $L \in \{1, 2, 3\}$
Monotone refinement: #cells non-decreasing in $L$.

In [6]:
P_wl = {L: wl_partition(_cora, depth=L) for L in (1, 2, 3)}
for L, P in P_wl.items():
    print(f'L={L}: {len(P.cells)} cells; max e_C = {max(P.e):.4f}; mean e_C = {float(np.mean(P.e)):.4f}')


L=1: 37 cells; max e_C = 0.7863; mean e_C = 0.3775
L=2: 1589 cells; max e_C = 0.7778; mean e_C = 0.0539
L=3: 2301 cells; max e_C = 0.8000; mean e_C = 0.0111


In [7]:
ncells = [len(P_wl[L].cells) for L in (1,2,3)]
assert ncells[0] <= ncells[1] <= ncells[2], f'M1: refinement not monotone: {ncells}'
# Multi-class Bayes error e_C = 1 - p_mode lives in [0, (K-1)/K]; binarisation min(e, 1-e) ∈ [0, 1/2] is what the bracket consumes.
for L, P in P_wl.items():
    assert all(0 <= e <= 1 - 1/K + 1e-12 for e in P.e), f'M1: bad e_C at L={L}'
    bin_e = [min(e, 1 - e) for e in P.e]
    assert all(0 <= e <= 0.5 + 1e-12 for e in bin_e), f'M1: binarised e_C out of [0,1/2] at L={L}'
    assert abs(sum(P.q) - 1.0) < 1e-12
print(f'[GATE OK] M1.wl: monotone refinement ncells={ncells}; multi-class e_C bounded, binarisation in [0,1/2]')


[GATE OK] M1.wl: monotone refinement ncells=[37, 1589, 2301]; multi-class e_C bounded, binarisation in [0,1/2]


## Histogram of per-cell $e_C$ for $L=2$

In [8]:
fig, ax = plt.subplots(figsize=(6,3.5))
ax.hist(P_wl[2].e, bins=30, weights=P_wl[2].q, color='C0', alpha=0.85)
ax.set_xlabel('per-cell Bayes error $e_C$'); ax.set_ylabel('mass $q_C$')
ax.set_title(f'M1 — 1-WL(L=2) per-cell error histogram ({len(P_wl[2].cells)} cells)')
_plots = _PROJECTS / 'capstone' / 'milestone1' / 'plots'; _plots.mkdir(exist_ok=True)
plt.tight_layout(); fig.savefig(_plots / 'm1_ehist.png', dpi=120); plt.show()


/var/folders/2d/bqg0c_7d7j7bpxcsdt0wm8zh0000gn/T/ipykernel_3405/4186109831.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); fig.savefig(_plots / 'm1_ehist.png', dpi=120); plt.show()


In [9]:
reflect.log('M1.partitions', f'label=7×0; wl(L=2) has {len(P_wl[2].cells)} cells, max e_C={max(P_wl[2].e):.3f}', 'HIGH')


**End of M1.**